# OsteoClassAI v2 · 02 Entrenamiento base

1. *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.
2. Ejecutar las celdas en orden. Los resultados quedan en Google Drive: `MyDrive/OsteoClassAI/runs/`.


In [ ]:
!nvidia-smi -L

In [ ]:
!git clone -q https://github.com/walterandres23/osteoclassai.git
%cd osteoclassai/v2
!pip -q install -e . timm kagglehub

## Datos: descarga desde Kaggle, auditoría y particiones

In [ ]:
import kagglehub, pathlib
path = pathlib.Path(kagglehub.dataset_download('shashwatwork/knee-osteoarthritis-dataset-with-severity'))
root = next(p.parent for p in path.rglob('train') if p.is_dir())
print(root, sorted(p.name for p in root.iterdir()))

In [ ]:
!python -m osteoclass.data --root "$root" --out outputs/data > outputs_data.log; tail -n 25 outputs_data.log

## Entrenamiento
ConvNeXt-Tiny preentrenado (ImageNet), 5 grados KL. Selección del mejor modelo solo con `val`; `test` se evalúa una vez al final. En una T4 tarda unos 30–40 min.

In [ ]:
RUN = 'convnext_tiny'
!python -m osteoclass.train --root "$root" --splits outputs/data/splits.csv --model $RUN --epochs 25 --out outputs/runs/$RUN

## Guardar en Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/OsteoClassAI/runs
!cp -r outputs/runs/$RUN outputs/data/audit.json /content/drive/MyDrive/OsteoClassAI/runs/
!ls -la /content/drive/MyDrive/OsteoClassAI/runs/$RUN